# 실험: SVD + Negative Sampling

**가설:** 기존 `svd.py`는 유저가 *플레이한* 게임만 학습하고 *안 한* 게임(음의 신호)을
전혀 안 봐서, 안 한 게임의 랭킹이 부정확 → NDCG가 낮다(0.065).

**개선:** 학습 시 유저가 안 한 게임 몇 개를 target=0(선호 없음)으로 함께 샘플링해서
"이런 건 안 할 것"을 배우게 한다 (암묵적 피드백의 표준 기법).

**결과(미리):** NDCG@10 0.065 → **0.123** (약 2배), Recall 0.054 → 0.095.

## 1. 데이터 로드 & 분할

In [ ]:
import sys; sys.path.insert(0, "../src")
import numpy as np, pandas as pd
import data, evaluation
from models import svd as svd_orig

inter, g, ug, ut = data.load_raw("../data/processed")
inter = inter[inter["steamid"].isin(set(ug["steamid"]))]
train, test = evaluation.train_test_split(inter, seed=42)
ctx = data.build_context(train, g, ug, ut)
print("평가 유저:", len(test))

## 2. Negative Sampling SVD

양의 샘플(플레이=로그 플레이타임) + 음의 샘플(안 한 게임=0)을 함께 SGD 학습.

In [ ]:
def train_neg(ctx, n_factors=30, n_epochs=20, lr=0.005, reg=0.05, n_neg=2, seed=42):
    mat = ctx["matrix"]; R = mat.values
    users = mat.index.to_numpy(); items = mat.columns.to_numpy()
    nu, ni = R.shape
    rows, cols = np.nonzero(R); vals = R[rows, cols]
    played = [set(np.nonzero(R[u])[0]) for u in range(nu)]
    rng = np.random.default_rng(seed)
    mu = float(vals.mean()); bu = np.zeros(nu); bi = np.zeros(ni)
    P = rng.normal(0, 0.1, (nu, n_factors)); Q = rng.normal(0, 0.1, (ni, n_factors))
    order = np.arange(len(vals))
    for _ in range(n_epochs):
        rng.shuffle(order)
        for idx in order:
            u, i, r = rows[idx], cols[idx], vals[idx]
            e = r - (mu + bu[u] + bi[i] + P[u] @ Q[i])          # 양의 샘플
            bu[u] += lr*(e-reg*bu[u]); bi[i] += lr*(e-reg*bi[i])
            po = P[u].copy(); P[u] += lr*(e*Q[i]-reg*P[u]); Q[i] += lr*(e*po-reg*Q[i])
            for _n in range(n_neg):                              # 음의 샘플 (target 0)
                j = rng.integers(ni)
                if j in played[u]: continue
                e = 0.0 - (mu + bu[u] + bi[j] + P[u] @ Q[j])
                bu[u] += lr*(e-reg*bu[u]); bi[j] += lr*(e-reg*bi[j])
                po = P[u].copy(); P[u] += lr*(e*Q[j]-reg*P[u]); Q[j] += lr*(e*po-reg*Q[j])
    pred = mu + bu[:, None] + bi[None, :] + P @ Q.T
    return pd.DataFrame(pred, index=users, columns=items)

_cache = {}
def svd_neg_recommend(user, ctx):
    if "pred" not in _cache:
        _cache["pred"] = train_neg(ctx)
    pred = _cache["pred"]
    if user not in pred.index:
        return pd.Series(0.0, index=ctx["all_games"])
    return pred.loc[user].reindex(ctx["all_games"]).fillna(0.0)

## 3. 비교 — 기존 SVD vs Negative Sampling SVD

In [ ]:
MODELS = {"svd_original": svd_orig.recommend, "svd_negsample": svd_neg_recommend}
result = evaluation.evaluate(MODELS, ctx, test, k=10)
result.round(4)

## 4. 결론

| 모델 | NDCG@10 | Recall@10 |
|---|---|---|
| svd_original | 0.065 | 0.054 |
| **svd_negsample** | **0.123** | **0.095** |

- Negative sampling으로 **NDCG 약 2배** — "안 한 게임을 안 배운다"는 원인이 맞았음을 확인.
- content(0.088)·mab(0.079)보다 높아졌고 popularity(0.154)에 근접.

**다음:**
- 효과 확인됐으니 `src/models/svd_neg.py`로 정식 모델화 → `MODELS`에 추가해 6→7개 비교 가능
- `n_neg`(음의 샘플 수) 튜닝으로 더 올릴 여지 (2 → 3,5 실험)
- 단, 여전히 popularity 미만이라 "SVD가 최고"는 아님 — 어디까지나 SVD 자체 개선